# 01 — Potsdam Data Verification & GSD Check

**Purpose:** verify the preprocessed Potsdam patch dataset (2,904 × 512²),
confirm the GSD that fixes `RESOLUTION_FACTOR`, and sanity-check labels.

**Needs:** Potsdam patch dataset attached
(`harish77718/ovrsis-potsdam-team1`). No GPU required.
The GSD check additionally needs an ORIGINAL Potsdam GeoTIFF —
run that one cell wherever the original tiles are available.

**Outputs:** verification printouts only — no files. The patches were
created previously; this notebook never modifies them.

In [ ]:
# Bootstrap: clone repo and add src/ to sys.path. Run every session.
import subprocess, sys
from pathlib import Path

if Path("/kaggle/input").exists():                 # Kaggle
    CLONE_DIR = Path("/kaggle/working/VC/rg-geoprompt-peft")
    if not CLONE_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/HarishDeepak/VC.git", str(CLONE_DIR)],
            check=True)
        print(f"✓ cloned → {CLONE_DIR}")
    else:
        print(f"✓ repo present → {CLONE_DIR}")
    sys.path.insert(0, str(CLONE_DIR / "src"))
else:                                              # local (VS Code)
    for cand in ["../src", "src"]:
        if Path(cand, "rg_geoprompt").exists():
            sys.path.insert(0, str(Path(cand).resolve()))
            break

from rg_geoprompt import paths
print(paths.describe())

## GSD check — decides RESOLUTION_FACTOR (2 vs 4)
`(0.1, 0.1)` → 10cm → factor 2. `(0.05, 0.05)` → 5cm → factor 4.
Update `constants.RESOLUTION_FACTOR` if it disagrees with the current value (2).

In [ ]:
# Run wherever an ORIGINAL Potsdam .tif is available (not the PNG patches).
from pathlib import Path
try:
    import rasterio
    tifs = list(Path("/kaggle/input").rglob("*.tif"))
    if not tifs:
        print("No .tif found — attach/point to an original Potsdam tile.")
    else:
        with rasterio.open(tifs[0]) as src:
            print(f"{tifs[0].name}: resolution = {src.res}")
            print("→ RESOLUTION_FACTOR =", 2 if abs(src.res[0] - 0.1) < 0.01 else 4)
except ImportError:
    print("rasterio not installed — %pip install rasterio, or run locally.")

## Patch count + tile inventory

In [ ]:
from collections import Counter
from rg_geoprompt.datasets import tile_id_from_stem
from rg_geoprompt import paths

stems = [f.stem for f in sorted(paths.IMG_DIR.glob("*.png"))]
tiles = Counter(tile_id_from_stem(s) for s in stems)
print(f"Total patches: {len(stems)}  (expected 2904)")
for t, n in sorted(tiles.items()):
    print(f"  tile {t}: {n} patches")

## Shape, dtype and label-value sanity check

In [ ]:
import numpy as np
from PIL import Image
from rg_geoprompt import paths

stem = stems[0]
img = np.array(Image.open(paths.IMG_DIR / f"{stem}.png"))
lbl = np.array(Image.open(paths.LABEL_DIR / f"{stem}.png"))
print(f"image {img.shape} {img.dtype} | label {lbl.shape} {lbl.dtype}")
print(f"label values: {sorted(np.unique(lbl).tolist())}  (allowed: 0-5, 255)")
assert img.shape[:2] == (512, 512) and lbl.shape == (512, 512)
assert set(np.unique(lbl)).issubset({0, 1, 2, 3, 4, 5, 255})
print("✓ shapes and label IDs OK")

## Label distribution (sample-based)

In [ ]:
import random
random.seed(0)
counts = Counter()
for stem in random.sample(stems, 200):
    lbl = np.array(Image.open(paths.LABEL_DIR / f"{stem}.png"))
    v, c = np.unique(lbl, return_counts=True)
    counts.update(dict(zip(v.tolist(), c.tolist())))
total = sum(counts.values())
from rg_geoprompt.constants import CLASS_NAMES
for cls in range(6):
    print(f"  [{cls}] {CLASS_NAMES[cls]:12s}: {100*counts.get(cls,0)/total:5.1f}%")
print(f"  [255] boundary    : {100*counts.get(255,0)/total:5.1f}%")

## Visual check incl. resolution-bridge preview
Left: original patch. Right: same patch after the bridge — this is
approximately what 20cm DOP20 will look like to the model.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torchvision.transforms.functional as TF
from rg_geoprompt.augment import resolution_bridge
from rg_geoprompt.utils import colorize_mask

stem = random.choice(stems)
img = Image.open(paths.IMG_DIR / f"{stem}.png").convert("RGB")
lbl = np.array(Image.open(paths.LABEL_DIR / f"{stem}.png"))
img_t = TF.to_tensor(img)
bridged = resolution_bridge(img_t)

fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
ax[0].imshow(img); ax[0].set_title("original")
ax[1].imshow(bridged.permute(1, 2, 0)); ax[1].set_title("resolution-bridged")
ax[2].imshow(colorize_mask(lbl)); ax[2].set_title("label (NOT bridged)")
for a in ax: a.axis("off")
plt.show()